<a href="https://colab.research.google.com/github/Jeevith252/ML_BASIC_PROJECT/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
text = "NLP ,, Session is... going on!!!"
clean = re.sub(r'[^a-zA-Z]' , '' ,text)
print(clean)

NLPSessionisgoingon


In [4]:
text = "The climate is beautiful"
tokens = text.split()
print(tokens)

['The', 'climate', 'is', 'beautiful']


In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [9]:
text = "The climate is beautiful"
tokens = word_tokenize(text)
filtered = [
    word for word in tokens
    if word.lower() not in stopwords.words("english")
]
print(filtered)

['climate', 'beautiful']


In [12]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
words = ["played" , "reading"]
for word in words:
  print(stemmer.stem(word))

play
read


In [13]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
print(lemmatizer.lemmatize("running", pos="v"))

[nltk_data] Downloading package wordnet to /root/nltk_data...


run


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
text = ["The climate is beautiful" , "It is raining"]
vector = CountVectorizer()

In [15]:
x = vector.fit_transform(text)
print(x.toarray())

[[1 1 1 0 0 1]
 [0 0 1 1 1 0]]


In [18]:
print(vector.get_feature_names_out())

['beautiful' 'climate' 'is' 'it' 'raining' 'the']


In [20]:
pip install gensim --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.6 MB/s eta 0:00:00


In [27]:
from gensim.models import Word2Vec



sentences = ["climate", "is", "beautiful"],["it", "is", "raining"]
model = Word2Vec(sentences,vector_size=20,window=2,min_count=1,workers=4)

print(model.wv["climate"])

[-0.00428278  0.01413282  0.02700714  0.03526328 -0.02851561  0.0092941
  0.03044432 -0.02399025 -0.0155363   0.03398815  0.00815738  0.00094959
  0.01736819  0.00108889  0.04809413  0.02530302 -0.04458695 -0.0352078
  0.00450728  0.03196267]


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

text = ["I love this phone","I hate this phone's feature"]

labels = [1, 0]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(text)
model = LogisticRegression()
model.fit(X, labels)

new_text = ["i like this phone"]
new_x = vectorizer.transform(new_text)
predictions = model.predict(new_x)
print(predictions)

[1]


# **IMDB Project**

In [30]:
!pip install -q nltk spacy xgboost
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [31]:
import pandas as pd
import re,string,nltk,spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score , classification_report ,confusion_matrix
nltk.download("punkt_tab")
nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm")



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Loading the dataset**

In [33]:
df = pd.read_csv("/content/IMDB Dataset.csv")

In [34]:
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [36]:
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [37]:
df.isnull().sum()

,0
review,0
sentiment,0


In [41]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
49995,False
49996,False
49997,False
49998,False


In [45]:
stop = stopwords.words("english")

def preprocess(text):
  text = text.lower()

  text = re.sub(r"<.*?>"," ",text)
  text = re.sub(r'http\S+|www|S+' , ' ' ,text)
  text = re.sub(r'[^a-z\s]',' ',text)

  tokens = word_tokenize(text)
  tokens = [t for t in tokens if t not in stop]

  doc = nlp(" ".join(tokens))

  lemmas = [tok.lemma_ for tok in doc]
  return ' '.join(lemmas)

In [ ]:
df["clean_review"] = df["review"].apply(preprocess)

df[["review" , "clean_review"]].head()

In [ ]:
df['label'] = df['sentiment'].map({'negative': 0 , "positive" : 1})

X_train , X_test ,y_train , y_test = train_test_split(df["clean_review"] , df["label"] ,test_size = 0.2 , random_state =42 ,startify = df["label"])

In [ ]:
tfidf = TfidfVectorizer(max_features = 5000)
X_train_t = tfidf.fit_transform(X_train)
X_test_t = tfidf.transform(X_test)
print(X_train_t.shape)

In [ ]:
lr = LogisticRegression(mx_iter = 1000)
lr.fit(X_train_t , y_train)
pred_lr = lr.predict(X_test_t)
print("LR Accuracy",accuracy_score(y_test , pred_lr))
print(confusion_matrix(y_test,pred_lr))
print(classification_report(y_test,pred_lr))


In [ ]:
xbg = XGBClassfier(n_estimators = 100 ,max_depth = 6,learning_rate = 0.1, eval_metric = "logloss", random_state =42)
xgb.fir(X_train_t,y_train)
xgb_pred = xgb.predict(X_test_t)
print("XGB accuracy :",accuracy_score(y_test,xgb_pred))
print(confusion_matrix(y_test,pred_xgb))
print(classification_report(y_test,pred_xgb))

In [ ]:
results = pd.DataFrame({
    "Model" : ["Logistic Regression" , "XGBoost"],
    "Accuracy" : [accuracy_score(y_test , pred_lr),
                  accuracy_score(y_test,pred_xgb)]
})

print(results)

In [ ]:
def predict_review(review,model):
  clean = preprocess(review)
  vec = tfidf.transform([clean])
  pred = model.predict(vec)[0]

  return "Positive" if pred == 1 else "Negative"

print(predict_review("This movie was aboslutely fantastic." ,lr))
print(predict_review("Worst movie I have ever watched." , xgb))